# Étape 3 — Modélisation
**Projet :** Génération de données synthétiques de sinistres pour la tarification en assurance  
**Dataset :** Insurance Claims Data — 58 592 polices  
**Auteurs :** Groupe ISFA 2025-2026

---

## Contenu
1. Justification de l'approche retenue
2. Imports et configuration
3. Chargement et split des données
4. Fonction d'évaluation
5. Baseline XGBoost
6. Génération CTGAN
7. Génération TVAE
8. Classifieur augmenté XGBoost + CTGAN
9. Classifieur augmenté XGBoost + TVAE
10. SMOTE — baseline d'augmentation classique
11. **Génération LLM — Ollama/phi3.5**
12. Tableau comparatif final
13. Résumé


## 1. Justification de l'approche retenue

Le projet mobilise les **deux approches** autorisées par les consignes :

### Approche 1 — ML/DL génératif (CTGAN + TVAE + XGBoost)

| Critère | LLM seul | CTGAN/TVAE |
|---------|----------|------------|
| Adapté aux données tabulaires | ❌ Conçu pour le texte | ✅ Conçu pour les données tabulaires |
| Coût computationnel | ❌ Très élevé | ✅ Maîtrisé en local |
| Performance sur données numériques | ❌ Non démontré | ✅ State-of-the-art (MIT) |
| Pertinence actuarielle | ❌ Faible | ✅ Directement applicable |

### Approche 2 — Architecture LLM (Ollama/phi3.5 — local et gratuit)

Le LLM enrichit les sinistres synthétiques avec des **descriptions textuelles professionnelles**, comme un expert en assurance le ferait.

### Pipeline complet

```
Données réelles (58 592 polices)
        │
        ├──► CTGAN ──► 5 000 sinistres synthétiques (chiffres)
        │                        │
        ├──► TVAE  ──► 5 000 sinistres synthétiques (chiffres)
        │                        │
        │              Ollama/phi3.5 ──► descriptions textuelles
        │                        │
        └──► XGBoost ◄───────────┘
                │
        Comparaison AUC / F1 / Recall
```


## 2. Imports et configuration

In [ ]:
import pandas as pd
import numpy as np
import os, pickle, json
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, f1_score, recall_score,
                             precision_score, confusion_matrix,
                             ConfusionMatrixDisplay)
import xgboost as xgb

try:
    from sdv.single_table import CTGANSynthesizer, TVAESynthesizer
    from sdv.metadata import SingleTableMetadata
    SDV_AVAILABLE = True
    print("✅ SDV disponible")
except ImportError:
    SDV_AVAILABLE = False
    print("⚠️  SDV non disponible — pip install sdv")

try:
    from imblearn.over_sampling import SMOTE
    SMOTE_AVAILABLE = True
    print("✅ imbalanced-learn disponible")
except ImportError:
    SMOTE_AVAILABLE = False
    print("⚠️  imbalanced-learn non disponible — pip install imbalanced-learn")

try:
    from openai import OpenAI
    OPENAI_AVAILABLE = True
    print("✅ openai disponible (pour Ollama)")
except ImportError:
    OPENAI_AVAILABLE = False
    print("⚠️  openai non disponible — pip install openai")

os.makedirs('../outputs/models', exist_ok=True)
os.makedirs('../outputs/synthetic', exist_ok=True)
os.makedirs('../outputs/llm', exist_ok=True)

N_SYNTHETIC  = 5000
EPOCHS_CTGAN = 300
EPOCHS_TVAE  = 300
RANDOM_STATE = 42
TEST_SIZE    = 0.2
N_LLM        = 10  # Nombre de descriptions LLM à générer

print(f"\n✓ Configuration OK")
print(f"  N_SYNTHETIC  = {N_SYNTHETIC}")
print(f"  EPOCHS_CTGAN = {EPOCHS_CTGAN}")
print(f"  EPOCHS_TVAE  = {EPOCHS_TVAE}")
print(f"  N_LLM        = {N_LLM}")

## 3. Chargement et split des données

- `data_preprocessed.csv` → pour le classifieur XGBoost (données normalisées)
- `data_encoded.csv` → pour les générateurs CTGAN/TVAE (échelle naturelle)
- `Insurance claims data.csv` → pour le contexte LLM (colonnes originales lisibles)

Le split est **stratifié** — le ratio 6.4% est conservé dans train et test.  
Le test set est **uniquement réel** — les synthétiques n'interviennent qu'en entraînement.

In [ ]:
df_encoded = pd.read_csv('../outputs/data_encoded.csv')
df_pre     = pd.read_csv('../outputs/data_preprocessed.csv')
df_raw     = pd.read_csv('../data/Insurance claims data.csv', sep=',')

print(f"data_encoded.csv      : {df_encoded.shape}")
print(f"data_preprocessed.csv : {df_pre.shape}")
print(f"Insurance claims data : {df_raw.shape}")

X = df_pre.drop(columns=['claim_status'])
y = df_pre['claim_status']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print(f"\nSplit stratifié (test={TEST_SIZE}) :")
print(f"  Train : {X_train.shape[0]:,} polices — {int(y_train.sum()):,} sinistres ({y_train.mean()*100:.1f}%)")
print(f"  Test  : {X_test.shape[0]:,} polices  — {int(y_test.sum()):,} sinistres ({y_test.mean()*100:.1f}%)")

## 4. Fonction d'évaluation

Métriques utilisées :
- **AUC-ROC** : capacité discriminante globale
- **F1-score** : équilibre précision/rappel
- **Recall** : % de vrais sinistres détectés — **priorité actuarielle**
- **Precision** : % de sinistres prédits qui sont réels

In [ ]:
results = []

def evaluate_model(model, X_test, y_test, model_name):
    y_pred      = model.predict(X_test)
    y_pred_prob = model.predict_proba(X_test)[:, 1]
    auc  = roc_auc_score(y_test, y_pred_prob)
    f1   = f1_score(y_test, y_pred, zero_division=0)
    rec  = recall_score(y_test, y_pred, zero_division=0)
    prec = precision_score(y_test, y_pred, zero_division=0)
    cm   = confusion_matrix(y_test, y_pred)

    print(f"\n── {model_name} ──")
    print(f"  AUC-ROC   : {auc:.4f}")
    print(f"  F1-score  : {f1:.4f}")
    print(f"  Recall    : {rec:.4f}  ← % de vrais sinistres détectés")
    print(f"  Precision : {prec:.4f}")
    print(f"  TN={cm[0,0]:>5}  FP={cm[0,1]:>5}")
    print(f"  FN={cm[1,0]:>5}  TP={cm[1,1]:>5}")

    fig, ax = plt.subplots(figsize=(4, 3))
    ConfusionMatrixDisplay(cm, display_labels=['Non-sinistre', 'Sinistre']).plot(
        ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f"{model_name}", fontsize=9, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"../outputs/figures/confusion_{model_name.replace(' ','_').replace('+','')}.png",
                dpi=120, bbox_inches='tight')
    plt.show(); plt.close()

    results.append({'model': model_name, 'auc': auc, 'f1': f1, 'recall': rec, 'precision': prec})

print("✓ Fonction d'évaluation définie")

## 5. Baseline — XGBoost sur données brutes

Point de référence obligatoire.  
`scale_pos_weight = 14` : une erreur sur un sinistre compte 14x plus qu'une erreur sur un non-sinistre.

In [ ]:
scale_pos_weight = int((y_train == 0).sum() / (y_train == 1).sum())
print(f"scale_pos_weight = {scale_pos_weight}")

xgb_baseline = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE, eval_metric='auc', verbosity=0
)
xgb_baseline.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
evaluate_model(xgb_baseline, X_test, y_test, "XGBoost Baseline")
pickle.dump(xgb_baseline, open('../outputs/models/xgb_baseline.pkl', 'wb'))
print("\n✓ Modèle sauvegardé")

## 6. Génération CTGAN

**Conditional Tabular GAN (MIT, 2019)**

Deux réseaux en compétition :
- Le **générateur** fabrique de faux profils de sinistres
- Le **discriminateur** essaie de détecter les faux

Après 300 epochs, le générateur produit des profils très réalistes.  
Entraîné **uniquement sur les sinistres** du train (2 998 polices).

In [ ]:
if SDV_AVAILABLE:
    df_train_full = X_train.copy()
    df_train_full['claim_status'] = y_train.values
    df_sinistres = df_train_full[df_train_full['claim_status'] == 1].drop(columns=['claim_status'])
    print(f"Entraînement CTGAN sur {len(df_sinistres):,} sinistres réels...")

    metadata = SingleTableMetadata()
    metadata.detect_from_dataframe(df_sinistres)

    ctgan = CTGANSynthesizer(metadata, epochs=EPOCHS_CTGAN, verbose=True)
    ctgan.fit(df_sinistres)
    print("\n✓ CTGAN entraîné")

    synthetic_ctgan = ctgan.sample(num_rows=N_SYNTHETIC)
    synthetic_ctgan['claim_status'] = 1
    synthetic_ctgan.to_csv('../outputs/synthetic/synthetic_ctgan.csv', index=False)
    ctgan.save('../outputs/models/ctgan_model.pkl')
    print(f"✓ {N_SYNTHETIC:,} sinistres synthétiques générés")
    print("\nAperçu :")
    display(synthetic_ctgan.head(3))
    CTGAN_AVAILABLE = True
else:
    print("⚠️  SDV non disponible")
    CTGAN_AVAILABLE = False

## 7. Génération TVAE

**Tabular Variational Autoencoder (MIT, 2019)**

Compression/décompression :
1. L'**encodeur** compresse chaque sinistre en quelques chiffres (espace latent)
2. Le **décodeur** reconstruit un sinistre depuis ces chiffres
3. Pour générer : on tire des chiffres aléatoires → décodeur → sinistre synthétique

**Avantage vs CTGAN** : plus stable, pas de risque de mode collapse.

In [ ]:
if SDV_AVAILABLE:
    print(f"Entraînement TVAE sur {len(df_sinistres):,} sinistres réels...")
    tvae = TVAESynthesizer(metadata, epochs=EPOCHS_TVAE, verbose=True)
    tvae.fit(df_sinistres)
    print("\n✓ TVAE entraîné")

    synthetic_tvae = tvae.sample(num_rows=N_SYNTHETIC)
    synthetic_tvae['claim_status'] = 1
    synthetic_tvae.to_csv('../outputs/synthetic/synthetic_tvae.csv', index=False)
    tvae.save('../outputs/models/tvae_model.pkl')
    print(f"✓ {N_SYNTHETIC:,} sinistres synthétiques générés")
    print("\nAperçu :")
    display(synthetic_tvae.head(3))
    TVAE_AVAILABLE = True
else:
    print("⚠️  SDV non disponible")
    TVAE_AVAILABLE = False

## 8. Classifieur augmenté — XGBoost + CTGAN

On mélange données réelles + synthétiques CTGAN.  
`scale_pos_weight` recalculé dynamiquement après augmentation.

In [ ]:
if 'CTGAN_AVAILABLE' in dir() and CTGAN_AVAILABLE:
    syn_X = synthetic_ctgan.drop(columns=['claim_status'], errors='ignore')
    syn_X = syn_X.reindex(columns=X_train.columns, fill_value=0)
    X_aug = pd.concat([X_train, syn_X], ignore_index=True)
    y_aug = pd.concat([y_train, pd.Series([1]*N_SYNTHETIC)], ignore_index=True)
    spw   = max(1, int((y_aug==0).sum() / (y_aug==1).sum()))

    print(f"Train augmenté : {len(X_aug):,} polices — {int(y_aug.sum()):,} sinistres ({y_aug.mean()*100:.1f}%)")
    print(f"scale_pos_weight ajusté : {spw}")

    xgb_ctgan = xgb.XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, scale_pos_weight=spw,
        random_state=RANDOM_STATE, eval_metric='auc', verbosity=0
    )
    xgb_ctgan.fit(X_aug, y_aug, eval_set=[(X_test, y_test)], verbose=False)
    evaluate_model(xgb_ctgan, X_test, y_test, "XGBoost + CTGAN")
    pickle.dump(xgb_ctgan, open('../outputs/models/xgb_ctgan.pkl', 'wb'))
    print("\n✓ Modèle sauvegardé")
else:
    print("⚠️  CTGAN non disponible")

## 9. Classifieur augmenté — XGBoost + TVAE

In [ ]:
if 'TVAE_AVAILABLE' in dir() and TVAE_AVAILABLE:
    syn_X = synthetic_tvae.drop(columns=['claim_status'], errors='ignore')
    syn_X = syn_X.reindex(columns=X_train.columns, fill_value=0)
    X_aug = pd.concat([X_train, syn_X], ignore_index=True)
    y_aug = pd.concat([y_train, pd.Series([1]*N_SYNTHETIC)], ignore_index=True)
    spw   = max(1, int((y_aug==0).sum() / (y_aug==1).sum()))

    print(f"Train augmenté : {len(X_aug):,} polices — {int(y_aug.sum()):,} sinistres ({y_aug.mean()*100:.1f}%)")
    print(f"scale_pos_weight ajusté : {spw}")

    xgb_tvae = xgb.XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, scale_pos_weight=spw,
        random_state=RANDOM_STATE, eval_metric='auc', verbosity=0
    )
    xgb_tvae.fit(X_aug, y_aug, eval_set=[(X_test, y_test)], verbose=False)
    evaluate_model(xgb_tvae, X_test, y_test, "XGBoost + TVAE")
    pickle.dump(xgb_tvae, open('../outputs/models/xgb_tvae.pkl', 'wb'))
    print("\n✓ Modèle sauvegardé")
else:
    print("⚠️  TVAE non disponible")

## 10. SMOTE — Baseline d'augmentation classique

SMOTE **interpole** entre des sinistres existants — il ne génère pas de nouvelles distributions.  
Inclus comme référence : si CTGAN/TVAE ne font pas mieux, le deep learning n'apporte rien.

In [ ]:
if SMOTE_AVAILABLE:
    smote = SMOTE(random_state=RANDOM_STATE)
    X_smote, y_smote = smote.fit_resample(X_train, y_train)
    spw = max(1, int((y_smote==0).sum() / (y_smote==1).sum()))

    print(f"Train SMOTE : {len(X_smote):,} polices — {int(y_smote.sum()):,} sinistres ({y_smote.mean()*100:.1f}%)")

    xgb_smote = xgb.XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, scale_pos_weight=spw,
        random_state=RANDOM_STATE, eval_metric='auc', verbosity=0
    )
    xgb_smote.fit(X_smote, y_smote, eval_set=[(X_test, y_test)], verbose=False)
    evaluate_model(xgb_smote, X_test, y_test, "XGBoost + SMOTE")
    pickle.dump(xgb_smote, open('../outputs/models/xgb_smote.pkl', 'wb'))
    print("\n✓ Modèle sauvegardé")
else:
    print("⚠️  imbalanced-learn non disponible")

## 11. Génération LLM — Ollama/phi3.5

### Architecture LLM locale

**Ollama** est un serveur LLM local — il fait tourner des modèles d'IA directement sur ta machine, sans envoyer de données sur internet et sans frais.

**phi3.5** est le modèle de Microsoft — petit, rapide et très performant pour la rédaction professionnelle.

### Ce que fait cette section

Pour chaque sinistre du dataset, le LLM génère une description textuelle professionnelle de 50 à 100 mots, comme un expert en assurance le rédigerait.

**Prérequis :**
- Ollama installé et en cours d'exécution (`ollama serve`)
- Modèle téléchargé (`ollama pull phi3.5`)
- Package openai installé (`pip install openai`)

In [ ]:
class ClaimsLLMGenerator:
    """Génère des descriptions de sinistres via LLM local (Ollama)."""

    def __init__(self):
        self.client = None
        self._load_model()

    def _load_model(self):
        if not OPENAI_AVAILABLE:
            print("⚠️  openai non installé : pip install openai")
            return
        try:
            from openai import OpenAI
            self.client = OpenAI(
                base_url="http://localhost:11434/v1",
                api_key="ollama"
            )
            # Test de connexion
            self.client.models.list()
            print("✅ Ollama API connectée (local — gratuit)")
        except Exception as e:
            print(f"⚠️  Ollama non accessible : {e}")
            print("   Vérifier qu'Ollama tourne en arrière-plan")
            self.client = None

    def create_prompt(self, claim_data) -> str:
        if isinstance(claim_data, str):
            details = claim_data
        else:
            details = "\n".join(f"- {k}: {v}" for k, v in claim_data.items())
        return f"""Tu es un expert en sinistres d'assurance automobile. Analyse ces données et rédige une description concise et précise du dossier:

DONNÉES DU DOSSIER:
{details}

Rédige une description professionnelle de 50-100 mots basée UNIQUEMENT sur les informations fournies:
"""

    def generate(self, claim_data) -> str:
        if self.client is None:
            return "LLM non disponible"
        try:
            response = self.client.chat.completions.create(
                model="phi3.5",
                messages=[
                    {"role": "system", "content": "You are a senior insurance claims expert with 20 years of experience."},
                    {"role": "user", "content": self.create_prompt(claim_data)}
                ],
                max_tokens=500,
                temperature=0.2,
                top_p=0.9
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            return f"Erreur : {e}"

    def generate_batch(self, claims_list) -> list:
        descriptions = []
        for i, claim in enumerate(claims_list):
            print(f"  Génération {i+1}/{len(claims_list)}...", end='\r')
            descriptions.append(self.generate(claim))
        print(f"\n✅ {len(descriptions)} descriptions générées")
        return descriptions

# Instanciation
llm = ClaimsLLMGenerator()
print(f"\nGénérateur LLM prêt : {llm.client is not None}")

In [ ]:
# Génération des descriptions
raw_cols = ['customer_age', 'vehicle_age', 'subscription_length',
            'region_density', 'fuel_type', 'segment',
            'transmission_type', 'airbags', 'ncap_rating']

df_claims_raw = df_raw[df_raw['claim_status'] == 1][raw_cols].head(N_LLM)
claims_list   = df_claims_raw.to_dict(orient='records')

if llm.client is not None:
    print(f"Génération de {N_LLM} descriptions de sinistres réels...\n")
    descriptions = llm.generate_batch(claims_list)

    # Sauvegarde
    df_llm = df_claims_raw.copy().reset_index(drop=True)
    df_llm['description_llm'] = descriptions
    df_llm.to_csv('../outputs/llm/sinistres_avec_descriptions.csv', index=False)

    # Affichage des 3 premiers exemples
    for i in range(min(3, len(descriptions))):
        print(f"\n{'='*55}")
        print(f"Sinistre {i+1} :")
        print(f"  Données : {claims_list[i]}")
        print(f"  Description LLM :")
        print(f"  {descriptions[i]}")

    print(f"\n✓ Sauvegardé : outputs/llm/sinistres_avec_descriptions.csv")
    display(df_llm.head())
else:
    print("⚠️  LLM non disponible — vérifier Ollama")
    print("   1. Ollama installé : https://ollama.com")
    print("   2. Modèle téléchargé : ollama pull phi3.5")
    print("   3. Serveur lancé : ollama serve")

## 12. Tableau comparatif final

Comparaison des 4 modèles sur le **même jeu de test réel** (11 719 polices, 750 sinistres).

**Lecture :**
- AUC proche de 1 → bonne discrimination globale
- Recall élevé → on détecte beaucoup de vrais sinistres ← priorité actuarielle

In [ ]:
df_results = pd.DataFrame(results).sort_values('f1', ascending=False)
df_results.to_csv('../outputs/resultats_modelisation.csv', index=False)

print("Tableau comparatif :")
display(df_results.round(4))

# Graphique
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
metrics = ['auc','f1','recall','precision']
labels  = ['AUC-ROC','F1-score','Recall','Precision']
colors  = ['#1F4E79','#2E86AB','#E74C3C','#2ECC71']

for i, (metric, label) in enumerate(zip(metrics, labels)):
    vals = df_results[metric]
    bars = axes[i].bar(df_results['model'], vals, color=colors[i], alpha=0.8, edgecolor='white')
    axes[i].set_title(label, fontweight='bold')
    axes[i].set_ylim(0, max(vals)*1.25 if max(vals) > 0 else 1)
    axes[i].tick_params(axis='x', rotation=30)
    for bar, val in zip(bars, vals):
        axes[i].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
                     f'{val:.3f}', ha='center', fontsize=8, fontweight='bold')

plt.suptitle("Comparaison des 4 modèles", fontsize=13, fontweight='bold', color='#1F4E79')
plt.tight_layout()
plt.savefig('../outputs/figures/09_comparaison_modeles.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n✓ Figure sauvegardée")

## 13. Résumé

In [ ]:
model_info = {
    'feature_columns': X.columns.tolist(),
    'n_features'     : len(X.columns),
    'test_size'      : TEST_SIZE,
    'n_synthetic'    : N_SYNTHETIC,
    'random_state'   : RANDOM_STATE
}
with open('../outputs/models/model_info.json', 'w') as f:
    json.dump(model_info, f, indent=2)

print("=" * 55)
print("  RÉSUMÉ ÉTAPE 3 — MODÉLISATION")
print("=" * 55)
print(f"  Approche ML/DL : CTGAN + TVAE + XGBoost + SMOTE")
print(f"  Approche LLM   : Ollama/phi3.5 (local, gratuit)")
print(f"  Synthétiques   : {N_SYNTHETIC:,} CTGAN + {N_SYNTHETIC:,} TVAE")
print(f"  LLM            : {N_LLM} descriptions générées")
print()
if len(results) > 0:
    best_f1  = df_results.iloc[0]
    best_auc = df_results.loc[df_results['auc'].idxmax()]
    print(f"  Meilleur F1   : {best_f1['model']} → F1={best_f1['f1']:.4f}, Recall={best_f1['recall']:.4f}")
    print(f"  Meilleur AUC  : {best_auc['model']} → AUC={best_auc['auc']:.4f}")
print()
print("  Fichiers produits :")
print("    outputs/models/xgb_*.pkl")
print("    outputs/models/ctgan_model.pkl, tvae_model.pkl")
print("    outputs/synthetic/synthetic_ctgan.csv, synthetic_tvae.csv")
print("    outputs/llm/sinistres_avec_descriptions.csv")
print("    outputs/resultats_modelisation.csv")
print()
print("✓ Étape 3 terminée.")